In [ ]:
import sys
import time
from pathlib import Path
import logging

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from tqdm import tqdm
import h5py

# ---------------------------------------------------------------------------
# Basic SNR / attack configuration
# ---------------------------------------------------------------------------

# Supported S-box types:
#   lut_ascon, lut_bilgin, lut_allouzi, lut_lu_4, lut_lu_5, lut_lu_6, lut_lu_7
sbox_type   = "lut_ascon"
tested_sbox = sbox_type

# Number of traces in the traceset
n_trc = 10_000

# Secret and IV (used later for reference / checks)
key_hex = "000102030405060708090A0B0C0D0E0F"
iv_hex  = "00001000808C0001"   # ASCON-128a IV
iv_int  = int(iv_hex, 16)

# SNR options
snr_all_bits    = False   # compute SNR for all bits
snr_single_bit  = True    # compute SNR only for one bit
target_bit      = 49
target_register = 0       # 0..4 for x0..x4
debug           = False
verbose         = True

# ---------------------------------------------------------------------------
# Project root detection
# ---------------------------------------------------------------------------

def find_project_root(start: Path, markers=("fusesoc.conf", ".dojo_root")) -> Path:
    current = start
    while current != current.parent:
        if any((current / m).exists() for m in markers):
            return current
        current = current.parent

    raise RuntimeError(
        f"Could not find project root (looked for markers: {markers}). "
        "Please ensure you are inside the Side-Channel-Dojo repository."
    )

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Notebook / interactive: start from CWD
    SCRIPT_DIR = Path.cwd()

DOJO_ROOT = find_project_root(SCRIPT_DIR)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

ASCON_PY_DIR = DOJO_ROOT / "sw" / "ciphers" / "ASCON_init_python"
SCA_DIR      = DOJO_ROOT / "sw" / "sca_scripts"
HW_DIR       = DOJO_ROOT / "hw"

# Base dirs for ASCON SW SCA
BASE_PLOT_DIR  = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "plot"
BASE_CACHE_DIR = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "cache"

# Traceset (HDF5) directory and file
TRACESET_DIR  = DOJO_ROOT / "sw" / "traceset" / "ASCON" / "sw"
TRACESET_FILE = TRACESET_DIR / f"ascon_opt32_{sbox_type}_{n_trc // 1000}k.h5"

# Per-S-box plot directory (also used for SNR plots)
PLOT_DIR = BASE_PLOT_DIR / sbox_type

# Ensure directories exist
BASE_PLOT_DIR.mkdir(parents=True, exist_ok=True)
BASE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
TRACESET_DIR.mkdir(parents=True, exist_ok=True)

# Make local modules importable
sys.path.insert(0, str(ASCON_PY_DIR))
sys.path.insert(0, str(SCA_DIR))

from analyzer.attack.ascon.xheep_ascon_cpa.ascon_first_round import ascon_first_round

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
)

logging.info(
    "SNR configuration - snr_all_bits=%s, snr_single_bit=%s, "
    "target_bit=%d, target_register=%d, debug=%s",
    snr_all_bits,
    snr_single_bit,
    target_bit,
    target_register,
    debug,
)

# ---------------------------------------------------------------------------
# Configuration printout
# ---------------------------------------------------------------------------

def _yn(flag: bool) -> str:
    return "yes" if flag else "no"

print("\n================= CONFIGURATION =================")
print(f"DOJO_ROOT                : {DOJO_ROOT}")
print()
print("Target")
print(f"  Cipher                 : ASCON")
print(f"  S-box implementation   : {tested_sbox}")
print(f"  Notebook scope         : SNR analysis on ASCON SW traces")
print()
print("Paths")
print(f"  Traceset file          : {TRACESET_FILE}")
print(f"  Plot dir               : {PLOT_DIR}")
print()
print("SNR configuration")
print(f"  Total traces (n_trc)   : {n_trc}")
print(f"  SNR all bits           : {_yn(snr_all_bits)}")
print(f"  SNR single bit         : {_yn(snr_single_bit)}")
print(f"  Target bit             : {target_bit}")
print(f"  Target register        : x{target_register}")
print("=================================================\n")
